# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python) 

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [1]:
import pandas as pd

In [2]:

orders = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv')
catalog = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv')
marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv')


In [3]:

for name, df in [('orders', orders), ('catalog', catalog), ('marketing', marketing)]:
    print(f"--- {name} ---")
    print(df.info())
    print(df.isnull().sum())
    print("duplicados:", df.duplicated().sum())
    print()

--- orders ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB
None
id_pedido               0
id_usuario              0
fecha_hora_pedido       0
pais                  300
dispositivo            20
fue

---

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas 

---

In [4]:
# 1. Fechas al formato correcto
orders['fecha_hora_pedido'] = pd.to_datetime(orders['fecha_hora_pedido'])
marketing['fecha'] = pd.to_datetime(marketing['fecha'])

# 2. Quitar duplicados exactos
orders = orders.drop_duplicates()

# 3. Estandarizar texto (mayúsculas y tildes inconsistentes)
orders['pais'] = orders['pais'].str.strip().str.title()
orders['categoria_producto'] = orders['categoria_producto'].replace({'Electronica': 'Electrónica'})

# 4. Quitar cantidades inválidas (negativas o outliers absurdos tipo 10,000/20,000)
orders = orders[(orders['cantidad'] > 0) & (orders['cantidad'] <= 100) | (orders['cantidad'].isna())]

# 5. Quitar filas con nulos en columnas clave para el análisis de revenue
orders = orders.dropna(subset=['cantidad', 'precio_unitario', 'monto_descuento',
                                 'nombre_producto', 'categoria_producto', 'pais'])

# Verificación rápida
print("Filas finales orders:", len(orders))
print(orders['pais'].unique())
print(orders['categoria_producto'].unique())

# Marketing: recuperar canal nulo desde id_campaña (patrón: canal_pais)
marketing['canal'] = marketing.apply(
    lambda r: r['id_campaña'].replace('_' + r['pais'], '') if pd.isnull(r['canal']) else r['canal'],
    axis=1
)

# Verificación
print("nulos en canal:", marketing['canal'].isnull().sum())  # debe dar 0
print(marketing['canal'].unique())

Filas finales orders: 24610
['Argentina' 'Mexico' 'Colombia']
['Moda' 'Electrónica' 'Hogar']
nulos en canal: 0
['organic' 'paid_search' 'social']


---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [5]:
# exportar datasets
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

---

In [6]:
# Revenue total (lo que efectivamente se cobró, ya con descuento aplicado)
revenue_total = orders['monto_total'].sum()

# Costo total: se cruza orders con catalog para traer el costo_unitario de cada producto
orders_costo = orders.merge(catalog[['nombre_producto', 'costo_unitario']], on='nombre_producto', how='left')
orders_costo['costo_linea'] = orders_costo['cantidad'] * orders_costo['costo_unitario']
costo_total = orders_costo['costo_linea'].sum()

# Gasto en marketing
gasto_marketing_total = marketing['gasto'].sum()

# Profit
gross_profit = revenue_total - costo_total          # margen antes de marketing
net_profit = gross_profit - gasto_marketing_total    # profit real del negocio

print(f"Revenue total: ${revenue_total:,.2f}")
print(f"Costo total (COGS): ${costo_total:,.2f}")
print(f"Gasto marketing total: ${gasto_marketing_total:,.2f}")
print(f"Gross profit (revenue - costo): ${gross_profit:,.2f}")
print(f"Net profit (revenue - costo - marketing): ${net_profit:,.2f}")
print(f"Margen neto: {net_profit/revenue_total*100:.2f}%")

Revenue total: $9,498,030.89
Costo total (COGS): $3,785,159.27
Gasto marketing total: $2,871,843.53
Gross profit (revenue - costo): $5,712,871.62
Net profit (revenue - costo - marketing): $2,841,028.09
Margen neto: 29.91%


In [7]:

ticket_promedio = orders['monto_total'].mean()
cantidad_promedio = orders['cantidad'].mean()

producto_top_unidades = orders.groupby('nombre_producto')['cantidad'].sum().idxmax()
producto_top_revenue = orders.groupby('nombre_producto')['monto_total'].sum().idxmax()

gasto_por_canal = marketing.groupby('canal')['gasto'].sum().sort_values(ascending=False)

print(f"Ticket promedio por orden: ${ticket_promedio:,.2f}")
print(f"Cantidad promedio de productos por orden: {cantidad_promedio:.2f}")
print(f"Producto más vendido (unidades): {producto_top_unidades}")
print(f"Producto más vendido (revenue): {producto_top_revenue}")
print("\nGasto en marketing por canal:")
print(gasto_por_canal)


Ticket promedio por orden: $385.94
Cantidad promedio de productos por orden: 1.51
Producto más vendido (unidades): Vacuum-Pro-Black
Producto más vendido (revenue): Vacuum-Pro-Black

Gasto en marketing por canal:
canal
social         976818.37
organic        972650.96
paid_search    922374.20
Name: gasto, dtype: float64


---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [8]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [9]:
query_eventos = '''
SELECT nombre_evento, COUNT(DISTINCT id_usuario) AS usuarios_unicos
FROM events
GROUP BY nombre_evento
ORDER BY usuarios_unicos DESC;
'''
eventos_resumen = pd.read_sql(query_eventos, con=engine)
eventos_resumen

,nombre_evento,usuarios_unicos
0,first_visit,7796
1,add_to_cart,7634
2,select_item,7582
3,begin_checkout,7208
4,add_payment_info,6250
5,purchase,6240


In [10]:

# PARTE 1: Totales del funnel
# ========================

query_totals = '''
SELECT nombre_evento, COUNT(DISTINCT id_usuario) AS usuarios_unicos
FROM events
GROUP BY nombre_evento
ORDER BY CASE nombre_evento
    WHEN 'first_visit' THEN 1
    WHEN 'select_item' THEN 2
    WHEN 'add_to_cart' THEN 3
    WHEN 'begin_checkout' THEN 4
    WHEN 'add_payment_info' THEN 5
    WHEN 'purchase' THEN 6
END;
'''

totals = pd.read_sql(query_totals, con=engine)
totals

,nombre_evento,usuarios_unicos
0,first_visit,7796
1,select_item,7582
2,add_to_cart,7634
3,begin_checkout,7208
4,add_payment_info,6250
5,purchase,6240


In [11]:

# PARTE 2: Conversiones
# ========================

query_conversion = '''
WITH funnel AS (
    SELECT
        nombre_evento,
        COUNT(DISTINCT id_usuario) AS usuarios_unicos,
        CASE nombre_evento
            WHEN 'first_visit' THEN 1
            WHEN 'select_item' THEN 2
            WHEN 'add_to_cart' THEN 3
            WHEN 'begin_checkout' THEN 4
            WHEN 'add_payment_info' THEN 5
            WHEN 'purchase' THEN 6
        END AS paso
    FROM events
    GROUP BY nombre_evento
)
SELECT
    nombre_evento,
    usuarios_unicos,
    paso,
    ROUND(100.0 * usuarios_unicos / LAG(usuarios_unicos) OVER (ORDER BY paso), 2) AS conversion_vs_paso_anterior,
    ROUND(100.0 * usuarios_unicos / FIRST_VALUE(usuarios_unicos) OVER (ORDER BY paso), 2) AS conversion_vs_inicio
FROM funnel
ORDER BY paso;
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion

,nombre_evento,usuarios_unicos,paso,conversion_vs_paso_anterior,conversion_vs_inicio
0,first_visit,7796,1,NaN,100.00
1,select_item,7582,2,97.26,97.26
2,add_to_cart,7634,3,100.69,97.92
3,begin_checkout,7208,4,94.42,92.46
4,add_payment_info,6250,5,86.71,80.17
5,purchase,6240,6,99.84,80.04


---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users` 
- `user_activity` 

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [12]:

# Explorar tabla user_activity
query_user_activity = '''
SELECT *
FROM user_activity;
'''

user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


In [13]:
# Retención por cohortes
query_cohort_retention_final = '''
WITH cohortes AS (
    SELECT
        id_usuario,
        DATE_TRUNC('month', CAST(fecha_registro AS DATE)) AS cohorte
    FROM users
),
tamano_cohorte AS (
    SELECT cohorte, COUNT(DISTINCT id_usuario) AS clientes_iniciales
    FROM cohortes
    GROUP BY cohorte
),
retencion AS (
    SELECT
        c.cohorte,
        COUNT(DISTINCT CASE WHEN ua.dias_despues_registro BETWEEN 1 AND 7 AND ua.activo = 1 THEN ua.id_usuario END) AS retenido_w1,
        COUNT(DISTINCT CASE WHEN ua.dias_despues_registro BETWEEN 8 AND 14 AND ua.activo = 1 THEN ua.id_usuario END) AS retenido_w2,
        COUNT(DISTINCT CASE WHEN ua.dias_despues_registro BETWEEN 15 AND 21 AND ua.activo = 1 THEN ua.id_usuario END) AS retenido_w3
    FROM cohortes c
    LEFT JOIN user_activity ua ON c.id_usuario = ua.id_usuario
    GROUP BY c.cohorte
)
SELECT
    t.cohorte,
    t.clientes_iniciales,
    r.retenido_w1,
    r.retenido_w2,
    r.retenido_w3,
    ROUND(100.0 * r.retenido_w1 / t.clientes_iniciales, 2) AS semana_1,
    ROUND(100.0 * r.retenido_w2 / t.clientes_iniciales, 2) AS semana_2,
    ROUND(100.0 * r.retenido_w3 / t.clientes_iniciales, 2) AS semana_3
FROM tamano_cohorte t
JOIN retencion r ON t.cohorte = r.cohorte
ORDER BY t.cohorte;
'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

,cohorte,clientes_iniciales,retenido_w1,retenido_w2,retenido_w3,semana_1,semana_2,semana_3
0,2025-01-01 00:00:00+00:00,1627,697,668,656,42.84,41.06,40.32
1,2025-02-01 00:00:00+00:00,1444,611,609,635,42.31,42.17,43.98
2,2025-03-01 00:00:00+00:00,1636,677,705,690,41.38,43.09,42.18
3,2025-04-01 00:00:00+00:00,1606,680,697,663,42.34,43.40,41.28
4,2025-05-01 00:00:00+00:00,1687,695,676,706,41.20,40.07,41.85


---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado** 
4. **Interpretar el resultado**  


---
Hipótesis estadística
   - **H₀ (H₀ (Hipótesis nula): No hay diferencia en la tasa de conversión entre control y tratamiento (p_control = p_tratamiento)** ...
   - **H₁ H₁ (Hipótesis alternativa): Sí hay diferencia en la tasa de conversión entre control y tratamiento (p_control ≠ p_tratamiento)** ...
   
**Test estadístico: Chi-cuadrado de independencia (apropiado porque convirtio es binaria y comparamos dos grupos independientes)** ...  
**Nivel de significancia alpha 0.05:** ...


In [14]:

experiment = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv')
print(experiment.shape)
print(experiment['variante'].value_counts())
print(experiment.groupby('variante')['convirtio'].mean())




(10000, 7)
tratamiento    5035
control        4965
Name: variante, dtype: int64
variante
control        0.156898
tratamiento    0.162860
Name: convirtio, dtype: float64


In [15]:
from scipy.stats import chi2_contingency

# Tabla de contingencia: variante vs convirtió/no convirtió
tabla = pd.crosstab(experiment['variante'], experiment['convirtio'])
print(tabla)

chi2, p_value, dof, expected = chi2_contingency(tabla)

print(f"\nChi2: {chi2:.4f}")
print(f"p-value: {p_value:.4f}")

alpha = 0.05
if p_value < alpha:
    print("Se rechaza H0: SÍ hay diferencia significativa en la conversión")
else:
    print("No se rechaza H0: no hay evidencia suficiente de diferencia significativa")

convirtio       0    1
variante              
control      4186  779
tratamiento  4215  820

Chi2: 0.6178
p-value: 0.4319
No se rechaza H0: no hay evidencia suficiente de diferencia significativa


---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión. 

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

---

## 🚀 Entrega Final

Comparte el acceso a tu Dashboard para revisión.   
Puedes entregar el Dashboard utilizando **Power BI o Tableau**.

Incluye **uno de los siguientes**:

- 🔗 Link público del dashboard publicado en **Power BI Service o Tableau Public / Tableau Cloud**
- 🔗 Link de **Google Drive o OneDrive** con el archivo del proyecto (`.pbix`) y los 3 csvs limpios.


### 📎 Enlace del Dashboard


https://drive.google.com/drive/folders/12IUXRvfN2ZpKshQr9471YNq9OpKLkGOH?usp=sharing